# Emma's card, one asset per modality

Run this notebook on your **laptop**. It makes one asset of each kind for Emma's card: an illustration, the same Emma in a new scene, a spoken greeting, a short jingle, and an animated clip. Each comes from a different open model on Nebius Serverless AI.

This is a follow-up to lesson 02. It assumes you have done `first_image_local.ipynb`, so you know how to create an Endpoint, copy its URL and token, and stop it.

Each section needs **its own Endpoint**. Work one section at a time: create the Endpoint, run the section, stop the Endpoint, then move on. Sections 2 and 5 reuse the illustration from section 1, so run section 1 first in the same kernel.

> An Endpoint bills from the moment it starts until you stop it, idle or not. Create each one right before its section, and stop it when the section's last cell has run.

Keep prompts free of real people, brands and known characters. Use preset voices only.

## Setup

`connect(name)` returns one Endpoint's HTTPS URL and token, the same way `setup_endpoint()` does in lesson 02. It reads `code/.env`, then the environment, and asks only for what is still missing. Each section has its own keys, for example `SANA_BASE_URL` and `SANA_TOKEN`.

The answers are saved to `code/.env` (readable only by you, and ignored by Git). The token is typed with `getpass`, so it is never echoed. Leave it empty if the Endpoint has no authentication. When you recreate an Endpoint, its URL and token change: delete that section's lines from `code/.env` and run its `connect` cell again.

Set `SAVE = True` to also write a smaller copy of each output (images as WebP, the clip re-encoded with H.264) and a `manifest.json` to `runs/emma-gift/` (ignored by Git). Re-running a section overwrites its file and its manifest record. With `SAVE = False` nothing is written.

In [ ]:
import base64
import hashlib
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from datetime import UTC, datetime
from getpass import getpass
from pathlib import Path

import certifi
import requests
from IPython.display import Audio, Image, Video, display
from PIL import Image as PILImage

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
found = [p for p in candidates if (p / "common.py").is_file()]
assert found, "Open this notebook from the repository (its code/ folder)"
CODE = found[0].resolve()
sys.path.insert(0, str(CODE))

from common import load_dotenv, save_dotenv, wait_ready  # noqa: E402

SAVE = True  # True: also write raw outputs to runs/emma-gift/ (overwrites)
ENV_FILE = CODE / ".env"
RUN_DIR = Path("runs/emma-gift")


def connect(name):
    """Return (base_url, token) for one section: code/.env, then the environment, then ask.

    Keys are <NAME>_BASE_URL and <NAME>_TOKEN. An empty token means no auth header.
    """
    url_key, token_key = f"{name}_BASE_URL", f"{name}_TOKEN"
    saved = load_dotenv(ENV_FILE)
    base = saved.get(url_key) or os.getenv(url_key) or ""
    token = saved.get(token_key, os.getenv(token_key))
    if not base:
        base = input(f"{name} Endpoint HTTPS URL: ").strip()
    if token is None:
        token = getpass(f"{name} Endpoint token (empty if auth is off): ").strip()
    base = base.strip().rstrip("/").removesuffix("/v1")
    assert base.startswith("https://"), "Paste the Endpoint's https:// URL"

    saved.update({url_key: base, token_key: token})
    save_dotenv(ENV_FILE, saved)
    print(f"{name} Endpoint configured" + (f" (***{token[-4:]})" if token else " (no auth)"))
    return base, token


def auth(token):
    return {"Authorization": f"Bearer {token}"} if token else {}


def show(data, kind):
    """Display bytes inline: kind is "image", "audio" or "video"."""
    if kind == "image":
        display(Image(data=data, width=512))
    elif kind == "audio":
        display(Audio(data=data))
    else:
        display(Video(data=data, embed=True, mimetype="video/mp4", width=640))


def smaller(data, ext):
    """Shrink a file before saving: PNG -> WebP q85, MP4 -> H.264 crf 28 (no audio). Others unchanged."""
    if ext == "png":
        buf = io.BytesIO()
        PILImage.open(io.BytesIO(data)).convert("RGB").save(buf, format="WEBP", quality=85)
        return buf.getvalue(), "webp", "webp q85"
    if ext == "mp4":
        assert shutil.which("ffmpeg"), (
            "Install ffmpeg to save the clip (macOS: brew install ffmpeg)"
        )
        with tempfile.TemporaryDirectory() as tmp:
            src, dst = Path(tmp) / "in.mp4", Path(tmp) / "out.mp4"
            src.write_bytes(data)
            subprocess.run(
                [
                    "ffmpeg",
                    "-y",
                    "-loglevel",
                    "error",
                    "-i",
                    str(src),
                    "-an",
                    "-c:v",
                    "libx264",
                    "-crf",
                    "28",
                    "-pix_fmt",
                    "yuv420p",
                    "-movflags",
                    "+faststart",
                    str(dst),
                ],
                check=True,
            )
            return dst.read_bytes(), "mp4", "libx264 crf 28, no audio"
    return data, ext, None


def keep(name, data, ext, record):
    """With SAVE = True, write a smaller copy and append a record to manifest.json. Otherwise do nothing.

    The variables in memory keep the full-quality output, so later sections still get it as input.
    """
    if not SAVE:
        return None
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    original_kb = len(data) / 1024
    data, ext, saved_as = smaller(data, ext)
    path = RUN_DIR / f"{name}.{ext}"
    path.write_bytes(data)
    record = {
        "asset": name,
        **record,
        "saved_as": saved_as,
        "utc_date": datetime.now(UTC).date().isoformat(),
        "utc_time": datetime.now(UTC).isoformat(timespec="seconds"),  # when it was generated
        "file": path.name,
        "bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
    }
    manifest = RUN_DIR / "manifest.json"
    records = json.loads(manifest.read_text()) if manifest.exists() else []
    records = [r for r in records if r["asset"] != name] + [record]  # re-running replaces it
    manifest.write_text(json.dumps(records, indent=2) + "\n")
    print(f"Saved {path} ({original_kb:.0f} KB -> {len(data) / 1024:.0f} KB)")
    return path


## 1. Illustration: Sana

Text to image. One JSON POST to `/v1/images/generations`, the same shape as lesson 02. The PNG comes back base64-encoded in `data[0].b64_json`.

- Template: [Sana](https://console.nebius.com/serverless/endpoint/create?template=sana)
- GPU: L40S
- Auth: token authentication is on in the template. Copy the token from the create form.

In [ ]:
SANA_URL, SANA_TOKEN = connect("SANA")
models = wait_ready(SANA_URL, SANA_TOKEN, deadline_s=600)
print("Served models:", [m.get("id") for m in models["data"]])

In [ ]:
SANA_GPU = "L40S"
illustration_body = {
    "model": "Efficient-Large-Model/Sana_1600M_1024px_diffusers",
    "prompt": (
        "A watercolor storybook illustration of a seven-year-old girl with curly red hair, "
        "a yellow winter coat and a blue scarf, stadngin on a snowy hill "
        "under a starry night sky, warm colors, no text"
        "she is watching the stars"
    ),
    "size": "512x512",
    "n": 1,
    "response_format": "b64_json",
    "num_inference_steps": 20,
    "seed": 7,
}

start = time.perf_counter()
response = requests.post(
    f"{SANA_URL}/v1/images/generations",
    headers=auth(SANA_TOKEN),
    json=illustration_body,
    timeout=300,
    verify=certifi.where(),
)
latency_s = time.perf_counter() - start
response.raise_for_status()
illustration_png = base64.b64decode(response.json()["data"][0]["b64_json"])
print(f"{latency_s:.1f} s, {len(illustration_png) / 1024:.0f} KB")

In [ ]:
show(illustration_png, "image")
keep(
    "1_dream_illustration",
    illustration_png,
    "png",
    {
        "model": illustration_body["model"],
        "route": "POST /v1/images/generations",
        "prompt": illustration_body["prompt"],
        "seed": illustration_body["seed"],
        "settings": {k: illustration_body[k] for k in ("size", "num_inference_steps")},
        "latency_s": round(latency_s, 2),
        "gpu": SANA_GPU,
    },
)

### Stop this Endpoint

Stop (or delete) this Endpoint in the console now, before you create the next one. It bills until it is stopped, idle or not.

## 2. New scene: Qwen-Image-Edit-2511

Image editing. The illustration from section 1 goes in as a file, with an instruction that moves Emma to a new scene and keeps her look. This is a multipart POST to `/v1/images/edits`, not JSON. The PNG comes back in `data[0].b64_json`.

- Template: [Qwen-Image-Edit-2511](https://console.nebius.com/serverless/endpoint/create?template=qwen-image-edit-2511)
- GPU: H100
- Auth: token authentication is on in the template.

In [ ]:
EDIT_URL, EDIT_TOKEN = connect("QWEN_EDIT")
models = wait_ready(EDIT_URL, EDIT_TOKEN, deadline_s=1200)
print("Served models:", [m.get("id") for m in models["data"]])

In [ ]:
if "illustration_png" not in globals():
    raise RuntimeError("Run section 1 first: this edit starts from Emma's illustration")

EDIT_GPU = "H100"
edit_form = {
    "model": "Qwen/Qwen-Image-Edit-2511",
    "prompt": (
        "Move the same girl indoors: she sits beside a decorated Christmas tree, holding the brass "
        "she has just unpacked the telescope from the christmas gift box"
        "telescope. Keep her face, hair, coat and scarf unchanged. Same watercolor storybook style, no text"
    ),
    "size": "1024x1024",
    "output_format": "png",
    "num_inference_steps": "20",
    "seed": "7",
}

start = time.perf_counter()
response = requests.post(
    f"{EDIT_URL}/v1/images/edits",
    headers=auth(EDIT_TOKEN),
    data=edit_form,
    files={"image": ("emma.png", illustration_png, "image/png")},
    timeout=600,
    verify=certifi.where(),
)
latency_s = time.perf_counter() - start
response.raise_for_status()
new_scene_png = base64.b64decode(response.json()["data"][0]["b64_json"])
print(f"{latency_s:.1f} s, {len(new_scene_png) / 1024:.0f} KB")

In [ ]:
show(new_scene_png, "image")
keep(
    "2_christmas_gift",
    new_scene_png,
    "png",
    {
        "model": edit_form["model"],
        "route": "POST /v1/images/edits",
        "prompt": edit_form["prompt"],
        "seed": int(edit_form["seed"]),
        "settings": {
            "size": edit_form["size"],
            "num_inference_steps": int(edit_form["num_inference_steps"]),
        },
        "latency_s": round(latency_s, 2),
        "gpu": EDIT_GPU,
    },
)

### 2b. Compose: two images in, one frame out

Same Endpoint as section 2, so run this before you stop it. The edit can take several input images: send the `image` field once per image, and refer to them in the prompt as image 1 and image 2, in the order you send them.

Here image 1 is the night scene from section 1 (the environment) and image 2 is the Christmas gift scene from section 2 (Emma with her telescope). The result is the first frame of the clip in section 5.

In [ ]:
for needed in ("illustration_png", "new_scene_png"):
    if needed not in globals():
        raise RuntimeError("Run sections 1 and 2 first: this edit combines their two images")

compose_form = {
    "model": "Qwen/Qwen-Image-Edit-2511",
    # "prompt": (
    #     "Put the girl and the brass telescope from image 2 on the snowy hill under the starry night sky from image 1 "
    #     "She looks at the yellow stars through the telescope. Keep her face, hair, coat and scarf "
    #     "unchanged. Same watercolor storybook style, no text"
    # ),
    "prompt": (
        "Place the girl and the brass telescope from image 2 on the snowy hill "
        "under the starry night sky from image 1. Change her pose so she is actively "
        "looking THROUGH the telescope: show her in side profile, with one eye "
        "pressed directly against the small eyepiece. Raise the telescope to eye "
        "level, with its larger front lens pointing diagonally upward toward the "
        "yellow stars. Her head tilts to align her eye with the telescope, and both "
        "hands support its barrel. The eyepiece must touch her eye, not rest below "
        "her face or against her chest. Preserve her facial features, hair, coat, "
        "and scarf, adjusting only her pose and head angle. "
        "Keep the same watercolor storybook style. No text."
    ),
    "size": "1024x1024",
    "output_format": "png",
    "num_inference_steps": "20",
    "seed": "6",
}

start = time.perf_counter()
response = requests.post(
    f"{EDIT_URL}/v1/images/edits",
    headers=auth(EDIT_TOKEN),
    data=compose_form,
    files=[  # same field name twice: image 1, then image 2 (order matters for the prompt)
        (
            "image",
            ("2_christmas_gift.png", new_scene_png, "image/png"),
        ),  # image 2: Emma with the telescope
    ],
    timeout=600,
    verify=certifi.where(),
)
latency_s = time.perf_counter() - start
response.raise_for_status()
composed_png = base64.b64decode(response.json()["data"][0]["b64_json"])
print(f"{latency_s:.1f} s, {len(composed_png) / 1024:.0f} KB")

In [ ]:
show(composed_png, "image")
keep(
    "2b_gift_under_stars",
    composed_png,
    "png",
    {
        "model": compose_form["model"],
        "route": "POST /v1/images/edits",
        "prompt": compose_form["prompt"],
        "input": ["1_dream_illustration", "2_christmas_gift"],
        "seed": int(compose_form["seed"]),
        "settings": {
            "size": compose_form["size"],
            "num_inference_steps": int(compose_form["num_inference_steps"]),
        },
        "latency_s": round(latency_s, 2),
        "gpu": EDIT_GPU,
    },
)

### Stop this Endpoint

Stop (or delete) this Endpoint in the console now, before you create the next one. It bills until it is stopped, idle or not.

## 3. Greeting: Kokoro-82M

Text to speech with a preset voice. JSON POST to `/v1/audio/speech`; the response body is the MP3 itself.

- Template: [Kokoro-82M](https://console.nebius.com/serverless/endpoint/create?template=kokoro-82m)
- GPU: L40S
- Auth: token authentication is on in the template.

In [ ]:
TTS_URL, TTS_TOKEN = connect("KOKORO")
models = wait_ready(TTS_URL, TTS_TOKEN, deadline_s=600)
print("Served models:", [m.get("id") for m in models["data"]])

In [ ]:
TTS_GPU = "L40S"
greeting_body = {
    "model": "kokoro",
    "input": (
        "Happy holidays, Emma! We heard you wished for a telescope. "
        "May all your nights be full of stars."
    ),
    "voice": "af_bella",
    "response_format": "mp3",
}

start = time.perf_counter()
response = requests.post(
    f"{TTS_URL}/v1/audio/speech",
    headers=auth(TTS_TOKEN),
    json=greeting_body,
    timeout=120,
    verify=certifi.where(),
)
latency_s = time.perf_counter() - start
response.raise_for_status()
greeting_mp3 = response.content
print(f"{latency_s:.1f} s, {len(greeting_mp3) / 1024:.0f} KB")

In [ ]:
show(greeting_mp3, "audio")
keep(
    "3_greeting",
    greeting_mp3,
    "mp3",
    {
        "model": "hexgrad/Kokoro-82M",
        "route": "POST /v1/audio/speech",
        "input": greeting_body["input"],
        "seed": None,
        "settings": {
            "voice": greeting_body["voice"],
            "response_format": greeting_body["response_format"],
        },
        "latency_s": round(latency_s, 2),
        "gpu": TTS_GPU,
    },
)

### Stop this Endpoint

Stop (or delete) this Endpoint in the console now, before you create the next one. It bills until it is stopped, idle or not.

## 4. Jingle: ACE-Step 1.5

Text to music. JSON POST to `/v1/audio/generations`; the request blocks until the audio is ready and the body is the MP3. `thinking=false` skips the language-model planning step, which is faster.

- Template: [ACE-Step 1.5](https://console.nebius.com/serverless/endpoint/create?template=ace-step-1-5)
- GPU: H100
- Auth: check that token authentication is on in the create form. If you turn it off, leave the token empty here.
- First boot downloads several GB of weights and can take **15 to 30+ minutes** before `/v1/models` answers. The wait below allows 40 minutes. The Endpoint bills during that wait.

In [ ]:
MUSIC_URL, MUSIC_TOKEN = connect("ACE_STEP")
models = wait_ready(MUSIC_URL, MUSIC_TOKEN, deadline_s=2400)
print("Served models:", [m.get("id") for m in models["data"]])

In [ ]:
MUSIC_GPU = "H100"
jingle_body = {
    "model": "ACE-Step/Ace-Step1.5",
    "prompt": "gentle holiday music box melody with soft sleigh bells, warm and cozy, instrumental",
    "task_type": "text2music",
    "thinking": False,
    "audio_duration": 8,
    "inference_steps": 8,
    "audio_format": "mp3",
}

start = time.perf_counter()
response = requests.post(
    f"{MUSIC_URL}/v1/audio/generations",
    headers=auth(MUSIC_TOKEN),
    json=jingle_body,
    timeout=900,
    verify=certifi.where(),
)
latency_s = time.perf_counter() - start
response.raise_for_status()
jingle_mp3 = response.content
print(f"{latency_s:.1f} s, {len(jingle_mp3) / 1024:.0f} KB")

In [ ]:
show(jingle_mp3, "audio")
keep(
    "4_jingle",
    jingle_mp3,
    "mp3",
    {
        "model": jingle_body["model"],
        "route": "POST /v1/audio/generations",
        "prompt": jingle_body["prompt"],
        "seed": None,
        "settings": {
            k: jingle_body[k]
            for k in ("task_type", "thinking", "audio_duration", "inference_steps")
        },
        "latency_s": round(latency_s, 2),
        "gpu": MUSIC_GPU,
    },
)

### Stop this Endpoint

Stop (or delete) this Endpoint in the console now, before you create the next one. It bills until it is stopped, idle or not.

## 5. Clip: Wan2.2-I2V-A14B

Image to video. The composed frame from section 2b (Emma with her telescope under the stars) becomes the first frame. Multipart POST to `/v1/videos/sync`; the source frame goes in the field `input_reference` (not `image`), and the body of the response is the MP4. The request holds the connection until the clip is done, which takes minutes.

- Template: [Wan2.2-I2V-A14B](https://console.nebius.com/serverless/endpoint/create?template=wan22-i2v-a14b)
- GPU: H100
- Auth: check that token authentication is on in the create form. If you turn it off, leave the token empty here.
- `flow_shift=12.0`, `boundary_ratio=0.875` and both guidance scales at `1.0` are the template's settings for 480p. Keep them.

In [ ]:
VIDEO_URL, VIDEO_TOKEN = connect("WAN22_I2V")
models = wait_ready(VIDEO_URL, VIDEO_TOKEN, deadline_s=1800)
print("Served models:", [m.get("id") for m in models["data"]])

In [ ]:
if "composed_png" not in globals():
    raise RuntimeError("Run section 2b first: the clip starts from the composed frame")

VIDEO_GPU = "H100"
clip_form = {
    "model": "Wan-AI/Wan2.2-I2V-A14B-Diffusers",
    "prompt": "The girl is watching the stars throught telecope, stars are sparkling. Christams tree on the left side is decorated with Christmas lights.",
    "size": "640x480",
    "num_frames": "81",  # about 5 s at 16 fps [verify on first run]
    # On a 504 or a timeout, drop to "49" (about 3 s) and run again.
    "fps": "16",
    "num_inference_steps": "20",
    "guidance_scale": "1.0",
    "guidance_scale_2": "1.0",
    "flow_shift": "12.0",
    "boundary_ratio": "0.875",
    "seed": "7",
}

start = time.perf_counter()
response = requests.post(
    f"{VIDEO_URL}/v1/videos/sync",
    headers=auth(VIDEO_TOKEN),
    data=clip_form,
    files={"input_reference": ("frame.png", composed_png, "image/png")},
    timeout=1800,
    verify=certifi.where(),
)
latency_s = time.perf_counter() - start
response.raise_for_status()
clip_mp4 = response.content
print(f"{latency_s:.1f} s, {len(clip_mp4) / 1024:.0f} KB")

In [ ]:
show(clip_mp4, "video")
keep(
    "5_watching_stars",
    clip_mp4,
    "mp4",
    {
        "model": clip_form["model"],
        "route": "POST /v1/videos/sync",
        "prompt": clip_form["prompt"],
        "input": "2b_gift_under_stars",
        "seed": int(clip_form["seed"]),
        "settings": {k: clip_form[k] for k in clip_form if k not in ("model", "prompt", "seed")},
        "latency_s": round(latency_s, 2),
        "gpu": VIDEO_GPU,
    },
)


### Stop this Endpoint

Stop (or delete) this Endpoint in the console now, before you create the next one. It bills until it is stopped, idle or not.

## Done

Check the console: none of the five Endpoints should be `RUNNING`. If `SAVE = True`, your files and `manifest.json` are in the folder printed by `keep()`.